# Minimal analysis dataset

Strip redundant EJSCREEN columns to avoid collinearity:
- One scale per concept (percentiles for env, one demographic index).
- No P_D2_*/P_D5_*, no raw/percentile duplicates, no RSEI when keeping disaggregated air.

Then **join EJI 2024** (Environmental Justice Index) on census tract GEOID to add EJI indices and health outcome percentiles.

**Output:** `../.data/analysis_minimal.csv`

In [10]:
from pathlib import Path
import pandas as pd

data_dir = Path("../.data")

csv_candidates = [
    data_dir / "EJSCREEN_2023_Tracts_StatePct_with_AS_CNMI_GU_VI.csv",
    Path("../EJSCREEN_2023_Tracts_StatePct_with_AS_CNMI_GU_VI.csv"),
]

In [11]:
minimal_columns = [
    "ID",
    "ST_ABBREV",
    "CNTY_NAME",
    "P_CANCER",
    "P_RESP",
    "P_PM25",
    "P_OZONE",
    "P_DSLPM",
    "P_PTRAF",
    "P_LDPNT",
    "P_PNPL",
    "P_PRMP",
    "P_PTSDF",
    "P_UST",
    "P_PWDIS",
    "DEMOGIDX_2",
]

In [12]:
csv_path = None
for p in csv_candidates:
    if p.exists():
        csv_path = p
        break
if csv_path is None:
    raise FileNotFoundError(
        "EJSCREEN CSV not found. Place EJSCREEN_2023_Tracts_StatePct_with_AS_CNMI_GU_VI.csv "
        "in project root or .data/"
    )
print(f"Loading {csv_path}...")
df = pd.read_csv(csv_path)

Loading ../.data/EJSCREEN_2023_Tracts_StatePct_with_AS_CNMI_GU_VI.csv...


In [13]:
available = [c for c in minimal_columns if c in df.columns]
missing = [c for c in minimal_columns if c not in df.columns]
if missing:
    print(f"Warning: columns not in CSV (skipped): {missing}")

out = df[available].copy()
out = out.dropna(subset=available)
print(f"Rows after dropna: {len(out):,} (dropped {len(df) - len(out):,})")

Rows after dropna: 81,001 (dropped 5,080)


In [14]:
data_dir.mkdir(parents=True, exist_ok=True)
output_path = data_dir / "analysis_minimal.csv"
out.to_csv(output_path, index=False)
print(f"Saved {len(out):,} rows × {len(available)} columns -> {output_path}")

Saved 81,001 rows × 16 columns -> ../.data/analysis_minimal.csv


## Join EJI 2024 (Environmental Justice Index)

EJI uses the same census tract GEOID (11-digit FIPS). We left-join a subset of EJI columns so the minimal analysis includes overall EJI rank, social vulnerability, environmental burden, and health outcome percentiles (EJI methodology).

In [15]:
# EJI 2024: columns to add (GEOID used for join)
eji_path = data_dir / "EJI_2024_United_States.csv"
eji_columns = [
    "GEOID",
    "RPL_EJI",  # Overall EJI percentile (0-1)
    "SPL_EJI",  # Overall EJI score
    "RPL_SER",  # Socioeconomic rank percentile
    "RPL_SVM",  # Social vulnerability percentile
    "RPL_EBM",  # Environmental burden percentile
    "RPL_EJI_CBM",  # Climate burden percentile
    "EPL_CANCER",  # Cancer prevalence percentile (EJI)
    "EPL_ASTHMA",  # Asthma prevalence percentile (EJI)
    "LOCATION",  # Tract label (e.g. "Census Tract 208.04; Autauga County; Alabama")
]
if not eji_path.exists():
    print(f"EJI file not found: {eji_path}. Skipping EJI join.")
    eji_df = None
else:
    print(f"Loading {eji_path}...")
    eji_full = pd.read_csv(eji_path)
    available_eji = [c for c in eji_columns if c in eji_full.columns]
    missing_eji = [c for c in eji_columns if c not in eji_full.columns]
    if missing_eji:
        print(f"Warning: EJI columns not found (skipped): {missing_eji}")
    eji_df = eji_full[available_eji].copy()
    eji_df["GEOID"] = eji_df["GEOID"].astype(str).str.strip()
    print(f"EJI rows: {len(eji_df):,}, columns to join: {available_eji}")

Loading ../.data/EJI_2024_United_States.csv...
EJI rows: 85,188, columns to join: ['GEOID', 'RPL_EJI', 'SPL_EJI', 'RPL_SER', 'RPL_SVM', 'RPL_EBM', 'RPL_EJI_CBM', 'EPL_CANCER', 'EPL_ASTHMA', 'LOCATION']


In [16]:
# Left-join EJI on 11-digit census tract GEOID
if eji_df is not None:
    out["GEOID"] = out["ID"].astype(str).str.zfill(11)
    eji_merge = eji_df.set_index("GEOID")
    out = out.set_index("GEOID").join(eji_merge, how="left").reset_index()
    matched = out["RPL_EJI"].notna().sum() if "RPL_EJI" in out.columns else 0
    print(
        f"Joined EJI: {matched:,} tracts matched, {len(out):,} total rows, {len(out.columns)} columns"
    )
else:
    print("EJI join skipped.")

Joined EJI: 80,129 tracts matched, 81,001 total rows, 26 columns


In [17]:
# Save final dataset (minimal + EJI columns)
out.to_csv(output_path, index=False)
print(f"Saved {len(out):,} rows × {len(out.columns)} columns -> {output_path}")

Saved 81,001 rows × 26 columns -> ../.data/analysis_minimal.csv


In [18]:
out.head()

,GEOID,ID,ST_ABBREV,CNTY_NAME,P_CANCER,P_RESP,P_PM25,P_OZONE,P_DSLPM,P_PTRAF,...,DEMOGIDX_2,RPL_EJI,SPL_EJI,RPL_SER,RPL_SVM,RPL_EBM,RPL_EJI_CBM,EPL_CANCER,EPL_ASTHMA,LOCATION
0,1001020100,1001020100,AL,Autauga,65.0,57.0,78.0,49.0,54.0,29.0,...,0.212172,0.5685,1.3991,0.3536,0.4541,0.3450,0.3476,0.5721,0.6239,Census Tract 201; Autauga County; Alabama
1,1001020200,1001020200,AL,Autauga,65.0,57.0,79.0,51.0,58.0,68.0,...,0.478788,0.4907,1.2619,0.4049,0.5811,0.2808,0.2725,0.3101,0.5367,Census Tract 202; Autauga County; Alabama
2,1001020300,1001020300,AL,Autauga,65.0,57.0,80.0,47.0,66.0,58.0,...,0.315208,0.5395,1.3477,0.3138,0.3974,0.3503,0.3205,0.6013,0.6239,Census Tract 203; Autauga County; Alabama
3,1001020400,1001020400,AL,Autauga,65.0,57.0,81.0,49.0,75.0,85.0,...,0.182536,0.3089,0.9678,0.1878,0.1899,0.3779,0.3434,0.8054,0.3565,Census Tract 204; Autauga County; Alabama
4,1001020501,1001020501,AL,Autauga,65.0,57.0,82.0,48.0,78.0,70.0,...,0.243095,0.0954,0.5449,0.1739,0.2263,0.3186,0.0290,0.6320,0.2258,Census Tract 205.01; Autauga County; Alabama
